# 3.2 Profil Efisiensi Komputasi, Analisis Jejak Memori (RAM), dan Kebutuhan Penyimpanan Model
**Dataset:** DemogPairs (10.800 citra, 2.160 citra uji held-out independen / Strict Subject-Disjoint Partitioning)  
**Tujuan Evaluasi Efisiensi Komputasi (Reviewer 2 #8 & Audit Metrik):**
1. **Analisis Deterministik Jejak Memori RAM Array Fitur:** Menghitung kapasitas memori array fitur representasi tensor NumPy `float32` ($N 	imes D 	imes 4\text{ bytes}$) menggunakan `.nbytes` untuk subset uji, latih, dan total dataset melintasi variasi dimensi Single-Domain (768-D), Dual-Domain (1.536-D), dan Tri-Domain (2.304-D), serta menyelaraskan konsistensi matematis antara satuan biner MiB ($1024^2\text{ B}$) dan desimal MB ($10^6\text{ B}$).
2. **Audit Pipeline Eksperimen Classifier Scikit-learn:** Merujuk langsung struktur pipeline pada notebook referensi `experiment/code/md/{gnb,lr,rf,svm}` untuk memverifikasi jalur model `models/clf_demogpairs_*.pkl`, fitur `features/demogpairs_vit-*.pkl`, dan hasil evaluasi resmi `results/*.json`.
3. **Pengukuran Jejak Memori OS dengan `psutil`:** Mengukur baseline RSS (*Resident Set Size*), peak RSS, dan pertambahan RAM aktual selama alokasi buffer fitur dan proses inferensi tanpa menggunakan bobot atau data sintetis.
4. **Estimasi Ukuran Objek Model In-Memory dengan `pympler.asizeof`:** Menilai ukuran objek model Scikit-learn di memori heap Python dan membandingkannya secara terpisah dengan ukuran berkas serialisasi di penyimpanan disk (`model_size_mb`).
5. **Reproduksi Mandiri Table XI & Table XII Naskah IEEE Access:** Membangun DataFrame ringkasan yang merefleksikan seluruh lima dimensi efisiensi komputasi yang dipublikasikan pada naskah.

In [1]:
import os
import sys
import json
import time
import psutil
import numpy as np
import pandas as pd
import joblib
import gc
import threading
from pympler import asizeof

# Pastikan direktori kerja aktif adalah direktori eksperimen code jika dipanggil dari root repositori
if os.path.basename(os.getcwd()) != 'code' and os.path.isdir('experiment/code'):
    os.chdir('experiment/code')
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Import modul utilitas eksperimen repositori
import utils as u

# Konfigurasi Tampilan Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

proc = psutil.Process(os.getpid())
print("Lingkungan evaluasi siap.")
print(f"Python Executable: {sys.executable}")
print(f"Process PID: {proc.pid}")
print(f"Baseline RSS Awal: {proc.memory_info().rss / (1024**2):.2f} MiB ({proc.memory_info().rss / (1000**2):.2f} MB)")
print("Spesifikasi Perangkat Keras: Intel Core i9-13900H | RAM 32 GB DDR5 | NVIDIA RTX 4060 8 GB VRAM")


Lingkungan evaluasi siap.
Python Executable: C:\Users\Rafy\miniconda3\envs\torch-gpu\python.exe
Process PID: 85912
Baseline RSS Awal: 765.05 MiB (802.21 MB)
Spesifikasi Perangkat Keras: Intel Core i9-13900H | RAM 32 GB DDR5 | NVIDIA RTX 4060 8 GB VRAM


---
## 1. Analisis Kapasitas Memori RAM Array Fitur Numerik (NumPy `.nbytes`, MiB vs MB)

Representasi fitur multi-domain disimpan dalam format presisi tunggal IEEE 754 (`np.float32`), di mana setiap elemen menempati **4 bytes**. Kapasitas memori dihitung secara deterministik dengan formula:

$$\text{RAM (Bytes)} = N \times D \times 4\text{ bytes}$$
$$\text{RAM (MiB)} = \frac{N \times D \times 4}{1.048.576} \quad (1\text{ MiB} = 1024^2\text{ B})$$
$$\text{RAM (MB)} = \frac{N \times D \times 4}{1.000.000} \quad (1\text{ MB} = 10^6\text{ B})$$

Di mana:
- $N$ = jumlah sampel citra ($N_{\text{test}} = 2.160$, $N_{\text{train}} = 8.640$, $N_{\text{total}} = 10.800$).
- $D$ = dimensionalitas vektor laten ($D=768$ untuk Single, $D=1.536$ untuk Dual, $D=2.304$ untuk Tri-Domain).

### Penjelasan Konsistensi Satuan (MiB vs MB):
- Pada publikasi IEEE Access, kolom **RAM Consumption (MB)** pada **Table XII** dan **Table XI** menggunakan konvensi biner standar sistem komputasi ($1\text{ MB} = 2^{20}\text{ bytes} = 1.048.576\text{ bytes} = 1\text{ MiB}$).
- Untuk Held-Out Test Set Tri-Domain ($2.160 \times 2.304$):
  - $\text{Bytes} = 2.160 \times 2.304 \times 4 = 19.906.560\text{ bytes}$.
  - Nilai biner: $19.906.560 / 1.048.576 = 18,9844\text{ MiB} \approx \mathbf{18,98\text{ MB}}$ (dilaporkan pada naskah).
  - Nilai desimal SI: $19.906.560 / 1.000.000 = \mathbf{19,91\text{ MB}}$.
- Untuk Dual-Domain ($2.160 \times 1.536$):
  - $\text{Bytes} = 2.160 \times 1.536 \times 4 = 13.271.040\text{ bytes}$.
  - Nilai biner: $13.271.040 / 1.048.576 = 12,65625\text{ MiB} \approx \mathbf{12,66\text{ MB}}$ (dilaporkan pada naskah).
  - Nilai desimal SI: $13.271.040 / 1.000.000 = \mathbf{13,27\text{ MB}}$.
- Tabel di bawah menyajikan kedua satuan secara berdampingan untuk transparansi penuh.

In [2]:
# Definisi parameter dataset dan skema fitur
subsets = {
    'Held-Out Test Set': 2160,
    'Training Set': 8640,
    'Entire Dataset': 10800
}

schemes = {
    'Single-Domain (Face / Emotion / Age)': 768,
    'Dual-Domain (Emotion + Face)': 1536,
    'Tri-Domain (Face + Emotion + Age)': 2304
}

array_ram_records = []

for s_name, n_samples in subsets.items():
    for f_name, dim in schemes.items():
        # Verifikasi langsung ukuran array menggunakan instance NumPy nyata
        dummy_array = np.zeros((n_samples, dim), dtype=np.float32)
        nbytes_actual = dummy_array.nbytes
        mib_actual = nbytes_actual / (1024 ** 2)
        mb_decimal = nbytes_actual / (1000 ** 2)
        
        array_ram_records.append({
            'Subset': s_name,
            'Sample Count (N)': n_samples,
            'Feature Configuration': f_name,
            'Dimension (D)': dim,
            'Bytes': nbytes_actual,
            'RAM Capacity (MiB, 1024^2)': round(mib_actual, 4),
            'RAM Capacity (MB Decimal, 10^6)': round(mb_decimal, 4),
            'Reported in Table XII (MB)': round(mib_actual, 2)
        })
        del dummy_array

df_array_ram = pd.DataFrame(array_ram_records)
print("=== TABEL KAPASITAS MEMORI ARRAY FITUR NUMPY DI RAM (.nbytes) ===")
print(df_array_ram.to_string(index=False))

=== TABEL KAPASITAS MEMORI ARRAY FITUR NUMPY DI RAM (.nbytes) ===
           Subset  Sample Count (N)                Feature Configuration  Dimension (D)    Bytes  RAM Capacity (MiB, 1024^2)  RAM Capacity (MB Decimal, 10^6)  Reported in Table XII (MB)
Held-Out Test Set              2160 Single-Domain (Face / Emotion / Age)            768  6635520                      6.3281                           6.6355                      6.3300
Held-Out Test Set              2160         Dual-Domain (Emotion + Face)           1536 13271040                     12.6562                          13.2710                     12.6600
Held-Out Test Set              2160    Tri-Domain (Face + Emotion + Age)           2304 19906560                     18.9844                          19.9066                     18.9800
     Training Set              8640 Single-Domain (Face / Emotion / Age)            768 26542080                     25.3125                          26.5421                     25.3100
    

---
## 2. Struktur Pipeline Classifier Scikit-learn & Audit Ketersediaan Model Biner

Berdasarkan implementasi eksperimen classifier pada repositori (`experiment/code/md/{gnb,lr,rf,svm}`):
1. Fitur ViT diekstraksi dan disimpan pada `features/demogpairs_vit-{face,emotion,age}.pkl`.
2. Model dilatih menggunakan `GridSearchCV` melalui fungsi `u.evaluate_models()`, kemudian diserialisasi via `u.save_object((trained_model, elapsed), model_file)`.
3. Jalur penyimpanan model resmi:
   - **Logistic Regression (Tri-Domain):** `models/clf_demogpairs_lr_vit-face-emotion-age_LogisticRegression.pkl`
   - **Support Vector Machine (Tri-Domain):** `models/clf_demogpairs_svm_vit-face-emotion-age_SVC.pkl`
   - **Random Forest (Tri-Domain):** `models/clf_demogpairs_rf_vit-face-emotion-age_RandomForestClassifier.pkl`
   - **Gaussian Naive Bayes (Dual-Domain):** `models/clf_demogpairs_gnb_vit-emotion-face_GaussianNB.pkl`
4. Hasil evaluasi empiris tersimpan dalam berkas terstruktur `results/demogpairs_*.json`.

### Ketentuan Integritas Data & Reproduksibilitas:
Sesuai arahan audit naskah:
- **Dilarang keras** menggunakan bobot acak atau model simulasi buatan (`np.random.randn`).
- Berkas biner `.pkl` (model dan fitur) dikecualikan dari Git melalui `.gitignore` demi menjaga efisiensi repositori.
- Jika berkas `.pkl` tidak ditemukan di disk lokal, notebook akan melaporkan status ini secara eksplisit dan merujuk data hasil pengukuran aktual dari `results/*.json` tanpa mengubah angka maupun hasil evaluasi terdahulu.

In [3]:
# Definisi konfigurasi 4 classifier acuan eksperimen
reference_models = {
    'LR Tri': {
        'classifier': 'Logistic Regression',
        'domain': 'Tri-Domain (Face + Emotion + Age)',
        'model_file': 'models/clf_demogpairs_lr_vit-face-emotion-age_LogisticRegression.pkl',
        'feature_files': [
            'features/demogpairs_vit-face.pkl',
            'features/demogpairs_vit-emotion.pkl',
            'features/demogpairs_vit-age.pkl'
        ],
        'result_file': 'results/demogpairs_lr_vit-face-emotion-age_LogisticRegression.json'
    },
    'SVM Tri': {
        'classifier': 'Support Vector Machine (SVC)',
        'domain': 'Tri-Domain (Face + Emotion + Age)',
        'model_file': 'models/clf_demogpairs_svm_vit-face-emotion-age_SVC.pkl',
        'feature_files': [
            'features/demogpairs_vit-face.pkl',
            'features/demogpairs_vit-emotion.pkl',
            'features/demogpairs_vit-age.pkl'
        ],
        'result_file': 'results/demogpairs_svm_vit-face-emotion-age_SVC.json'
    },
    'RF Tri': {
        'classifier': 'Random Forest Classifier',
        'domain': 'Tri-Domain (Face + Emotion + Age)',
        'model_file': 'models/clf_demogpairs_rf_vit-face-emotion-age_RandomForestClassifier.pkl',
        'feature_files': [
            'features/demogpairs_vit-face.pkl',
            'features/demogpairs_vit-emotion.pkl',
            'features/demogpairs_vit-age.pkl'
        ],
        'result_file': 'results/demogpairs_rf_vit-face-emotion-age_RandomForestClassifier.json'
    },
    'GNB Dual': {
        'classifier': 'Gaussian Naive Bayes',
        'domain': 'Dual-Domain (Emotion + Face)',
        'model_file': 'models/clf_demogpairs_gnb_vit-emotion-face_GaussianNB.pkl',
        'feature_files': [
            'features/demogpairs_vit-face.pkl',
            'features/demogpairs_vit-emotion.pkl'
        ],
        'result_file': 'results/demogpairs_gnb_vit-emotion-face_GaussianNB.json'
    }
}

# Audit ketersediaan file biner di disk lokal
audit_records = []
for key, conf in reference_models.items():
    m_exists = os.path.exists(conf['model_file'])
    feats_exist = all(os.path.exists(f) for f in conf['feature_files'])
    res_exists = os.path.exists(conf['result_file'])
    
    audit_records.append({
        'Model Key': key,
        'Classifier': conf['classifier'],
        'Model File (.pkl)': conf['model_file'],
        'Model On Disk': m_exists,
        'Features On Disk': feats_exist,
        'Result JSON On Disk': res_exists
    })

df_audit = pd.DataFrame(audit_records)
print("=== STATUS AUDIT KETERSEDIAAN BERKAS DI DISK ===")
print(df_audit[['Model Key', 'Classifier', 'Model On Disk', 'Features On Disk', 'Result JSON On Disk']].to_string(index=False))

all_models_on_disk = all(r['Model On Disk'] for r in audit_records)
if all_models_on_disk:
    print("\n[STATUS ASSET BINARY: LENGKAP]")
    print("Informasi: Seluruh berkas model biner dan representasi fitur terverifikasi lengkap di disk lokal.")
    print("Pengukuran in-memory via pympler dan pengukuran dinamika RAM via psutil dieksekusi secara riil pada model fisik.")
else:
    print("\n[DIAGNOSTIK KETERSEDIAAN ASSET BINARY]")
    print("Informasi: Direktori 'models/' dan 'features/' tidak lengkap di disk lokal.")


=== STATUS AUDIT KETERSEDIAAN BERKAS DI DISK ===
Model Key                   Classifier  Model On Disk  Features On Disk  Result JSON On Disk
   LR Tri          Logistic Regression           True              True                 True
  SVM Tri Support Vector Machine (SVC)           True              True                 True
   RF Tri     Random Forest Classifier           True              True                 True
 GNB Dual         Gaussian Naive Bayes           True              True                 True

[STATUS ASSET BINARY: LENGKAP]
Informasi: Seluruh berkas model biner dan representasi fitur terverifikasi lengkap di disk lokal.
Pengukuran in-memory via pympler dan pengukuran dinamika RAM via psutil dieksekusi secara riil pada model fisik.


---
## 3. Profiling Alokasi Memori Fisik RAM (`psutil`) & Estimasi Ukuran Objek Model (`pympler.asizeof`)

### Metodologi Pengukuran:
1. **`psutil.Process().memory_info().rss`**:
   Mengukur *Resident Set Size* (RSS), yaitu memori fisik riil yang dialokasikan oleh sistem operasi untuk proses Python aktif. Kami mengukur:
   - **Baseline RSS**: Status memori proses sebelum eksekusi.
   - **Peak RSS**: Alokasi maksimum yang tercatat selama eksekusi.
   - **RAM Delta**: Selisih memori akibat alokasi buffer input dan komputasi inferensi.
2. **`pympler.asizeof.asizeof()`**:
   Mengukur ukuran objek model Scikit-learn aktif di heap Python secara mendalam (*deep recursive traversal*), mencakup seluruh atribut internal seperti koefisien, vektor pendukung (*support vectors*), struktur pohon keputusan (*decision trees*), dan kamus parameter.
3. **Pemisahan Konseptual `model_size_mb`**:
   `model_size_mb` adalah ukuran berkas serialisasi terkompresi di media penyimpanan disk (`joblib.dump(..., compress=9)`), sedangkan RAM array dan heap `asizeof` adalah jejak memori dinamis di RAM saat inferensi.

In [4]:
# 1. Demonstrasi Pengukuran Alokasi RAM Aktual pada Array Fitur Held-Out Test Set
process = psutil.Process(os.getpid())

rss_baseline = process.memory_info().rss
test_n = 2160
tri_dim = 2304

# Alokasi buffer data uji Tri-Domain nyata
test_features_buffer = np.ones((test_n, tri_dim), dtype=np.float32)

rss_allocated = process.memory_info().rss
ram_delta_bytes = rss_allocated - rss_baseline
ram_delta_mib = ram_delta_bytes / (1024 ** 2)
ram_delta_mb = ram_delta_bytes / (1000 ** 2)

print("=== VERIFIKASI PENGUKURAN RAM PROSES DENGAN PSUTIL ===")
print(f"Baseline Process RSS: {rss_baseline / (1024**2):.2f} MiB ({rss_baseline / (1000**2):.2f} MB)")
print(f"Process RSS Pasca-Alokasi Array Data Uji: {rss_allocated / (1024**2):.2f} MiB ({rss_allocated / (1000**2):.2f} MB)")
print(f"Delta Alokasi RAM Aktual: {ram_delta_mib:.2f} MiB ({ram_delta_mb:.2f} MB)")
print(f"NumPy Array .nbytes: {test_features_buffer.nbytes} bytes ({test_features_buffer.nbytes / (1024**2):.4f} MiB)")
print(f"Konsistensi: Delta alokasi RSS proses mencerminkan kapasitas array .nbytes (~18,98 MiB).")

# Hapus buffer setelah pengukuran selesai
del test_features_buffer

=== VERIFIKASI PENGUKURAN RAM PROSES DENGAN PSUTIL ===
Baseline Process RSS: 766.03 MiB (803.24 MB)
Process RSS Pasca-Alokasi Array Data Uji: 785.02 MiB (823.15 MB)
Delta Alokasi RAM Aktual: 18.99 MiB (19.91 MB)
NumPy Array .nbytes: 19906560 bytes (18.9844 MiB)
Konsistensi: Delta alokasi RSS proses mencerminkan kapasitas array .nbytes (~18,98 MiB).


In [5]:
# 2. Fungsi Eksekusi Pengukuran Classifier Menggunakan Model Aktual (Jika Ada) atau Fallback Log Resmi
data = u.load_demogpairs()
train_idx, test_idx = u.split_demogpairs_subject_disjoint(data)
test_data = [data[i] for i in test_idx]

def profile_classifier(conf_key, conf):
    m_path = conf['model_file']
    r_path = conf['result_file']
    feat_paths = conf['feature_files']
    
    # Baca hasil eksperimen resmi dari JSON untuk mempertahankan integritas data
    with open(r_path, 'r') as fp:
        res = json.load(fp)
        
    serialized_size_mb = res.get('model_size_mb', 0.0)
    latency_ms = res.get('latency_ms_per_sample', 0.0)
    train_time_sec = res.get('train_time_seconds', 0.0)
    acc = res.get('test_accuracy', 0.0) * 100.0
    f1 = res.get('test_f1', 0.0) * 100.0
    
    # RAM consumption array fitur held-out test set
    ram_array_mb = 18.98 if 'Tri' in conf_key else 12.66
    
    in_memory_obj_size_mb = None
    in_memory_best_mb = None
    baseline_rss_mb = None
    peak_rss_mb = None
    predict_rss_delta_mb = None
    peak_rss_delta_mb = None
    
    # Jika model biner aktual tersedia di disk, lakukan pengukuran pympler & psutil langsung
    if os.path.exists(m_path):
        print(f'[{conf_key}] Memuat model aktual dari {m_path}...')
        loaded_obj = u.load_object(m_path)
        model_obj = loaded_obj[0] if isinstance(loaded_obj, tuple) else loaded_obj
        
        # Hitung ukuran objek in-memory via pympler.asizeof
        in_memory_bytes = asizeof.asizeof(model_obj)
        in_memory_obj_size_mb = in_memory_bytes / (1024 ** 2)
        
        if hasattr(model_obj, 'best_estimator_'):
            in_memory_best_bytes = asizeof.asizeof(model_obj.best_estimator_)
            in_memory_best_mb = in_memory_best_bytes / (1024 ** 2)
        else:
            in_memory_best_mb = in_memory_obj_size_mb
        
        # Jika file fitur tersedia, muat X_test aktual dan ukur baseline, peak, dan delta RSS selama predict
        if all(os.path.exists(f) for f in feat_paths):
            feats_dict_list = [joblib.load(f) for f in feat_paths]
            X_test_list = []
            for d in test_data:
                key = d['image_path']
                vecs = [fd[key] for fd in feats_dict_list]
                X_test_list.append(np.concatenate(vecs) if len(vecs) > 1 else vecs[0])
            X_test = np.array(X_test_list, dtype=np.float32)
            
            p = psutil.Process(os.getpid())
            gc.collect()
            
            rss_before = p.memory_info().rss
            peak_rss = rss_before
            stop_monitor = False
            
            def monitor_memory():
                nonlocal peak_rss
                while not stop_monitor:
                    cur_rss = p.memory_info().rss
                    if cur_rss > peak_rss:
                        peak_rss = cur_rss
                    time.sleep(0.0005)
            
            mon_thread = threading.Thread(target=monitor_memory)
            mon_thread.daemon = True
            mon_thread.start()
            
            try:
                preds = model_obj.predict(X_test)
            finally:
                stop_monitor = True
                mon_thread.join()
                
            rss_after = p.memory_info().rss
            
            baseline_rss_mb = rss_before / (1024 ** 2)
            peak_rss_mb = peak_rss / (1024 ** 2)
            predict_rss_delta_mb = (rss_after - rss_before) / (1024 ** 2)
            peak_rss_delta_mb = (peak_rss - rss_before) / (1024 ** 2)
    else:
        # Jika model biner tidak tersedia di PC kantor, laporkan status tanpa data sintetis
        pass
        
    return {
        'Key': conf_key,
        'Classifier': res.get('model_name'),
        'Configuration': conf['domain'].split('(')[1].replace(')', ''),
        'Accuracy (%)': round(acc, 2),
        'F1-Score (%)': round(f1, 2),
        'Train Time (min)': round(train_time_sec / 60.0, 2),
        'Latency (ms/sample)': round(latency_ms, 4),
        'RAM Array Table XII (MB)': ram_array_mb,
        'Model Storage Disk (MB)': round(serialized_size_mb, 2),
        'In-Memory Grid (MiB)': round(in_memory_obj_size_mb, 2) if in_memory_obj_size_mb is not None else 'N/A',
        'Best Estimator Heap (MiB)': round(in_memory_best_mb, 2) if in_memory_best_mb is not None else 'N/A',
        'Baseline RSS (MiB)': round(baseline_rss_mb, 2) if baseline_rss_mb is not None else 'N/A',
        'Peak RSS (MiB)': round(peak_rss_mb, 2) if peak_rss_mb is not None else 'N/A',
        'Peak Delta RSS (MiB)': round(peak_rss_delta_mb, 2) if peak_rss_delta_mb is not None else 'N/A'
    }

classifier_profiles = [profile_classifier(k, c) for k, c in reference_models.items()]
df_profiles = pd.DataFrame(classifier_profiles)
print('=== HASIL PROFIL PENGUKURAN RIIL CLASSIFIER (PYMPLER & PSUTIL) ===')
display_cols = ['Classifier', 'Configuration', 'Accuracy (%)', 'F1-Score (%)', 'Train Time (min)', 'Latency (ms/sample)', 'RAM Array Table XII (MB)', 'Model Storage Disk (MB)', 'In-Memory Grid (MiB)', 'Best Estimator Heap (MiB)', 'Peak RSS (MiB)', 'Peak Delta RSS (MiB)']
print(df_profiles[display_cols].to_string(index=False))


[LR Tri] Memuat model aktual dari models/clf_demogpairs_lr_vit-face-emotion-age_LogisticRegression.pkl...


[SVM Tri] Memuat model aktual dari models/clf_demogpairs_svm_vit-face-emotion-age_SVC.pkl...


[RF Tri] Memuat model aktual dari models/clf_demogpairs_rf_vit-face-emotion-age_RandomForestClassifier.pkl...


[GNB Dual] Memuat model aktual dari models/clf_demogpairs_gnb_vit-emotion-face_GaussianNB.pkl...


=== HASIL PROFIL PENGUKURAN RIIL CLASSIFIER (PYMPLER & PSUTIL) ===
            Classifier        Configuration  Accuracy (%)  F1-Score (%)  Train Time (min)  Latency (ms/sample)  RAM Array Table XII (MB)  Model Storage Disk (MB)  In-Memory Grid (MiB)  Best Estimator Heap (MiB)  Peak RSS (MiB)  Peak Delta RSS (MiB)
    LogisticRegression Face + Emotion + Age       88.6600       88.6800           54.7800               0.0046                   18.9800                   0.1100                0.2000                     0.1000        960.0600               23.1400
                   SVC Face + Emotion + Age       88.6100       88.6300          511.1000               4.6785                   18.9800                  36.6000               73.7100                    73.4400       1072.3100               57.4600
RandomForestClassifier Face + Emotion + Age       85.0000       84.9300          110.4700               0.0334                   18.9800                   6.1900                0.4500   

---
## 4. Ekstraksi Empiris Seluruh Model Eksperimen dari Direktori `results/`

Repositori ini menyimpan 28 berkas log JSON hasil evaluasi empiris lengkap untuk seluruh kombinasi 4 classifier (LR, SVM, RF, GNB) dan 7 skema konfigurasi domain fitur (*face, emotion, age, face-emotion, emotion-face, face-age, emotion-age, face-emotion-age*).

Di bawah ini diekstraksi seluruh hasil evaluasi empiris untuk memvalidasi konsistensi metrik `test_accuracy`, `test_f1`, `train_time_seconds`, `latency_ms_per_sample`, dan `model_size_mb`.

In [6]:
results_dir = 'results'
result_files = [f for f in os.listdir(results_dir) if f.endswith('.json')]

all_records = []
for fname in sorted(result_files):
    fpath = os.path.join(results_dir, fname)
    with open(fpath, 'r') as fp:
        d = json.load(fp)
    
    parts = fname.replace('.json', '').split('_')
    clf = parts[1].upper()
    feat = '_'.join(parts[2:-1])
    
    # Hitung RAM array held-out test set berdasarkan dimensionalitas fitur
    if 'face-emotion-age' in feat:
        dim = 2304
        ram_mb = 18.98
    elif any(d2 in feat for d2 in ['emotion-face', 'face-emotion', 'face-age', 'emotion-age']):
        dim = 1536
        ram_mb = 12.66
    else:
        dim = 768
        ram_mb = 6.33
        
    all_records.append({
        'Classifier': clf,
        'Feature Domain': feat,
        'Dimension': dim,
        'Accuracy (%)': round(d.get('test_accuracy', 0.0) * 100.0, 2),
        'F1-Score (%)': round(d.get('test_f1', 0.0) * 100.0, 2),
        'Train Time (min)': round(d.get('train_time_seconds', 0.0) / 60.0, 2),
        'Latency (ms)': round(d.get('latency_ms_per_sample', 0.0), 4),
        'RAM (MB)': ram_mb,
        'Model Size (MB)': round(d.get('model_size_mb', 0.0), 2)
    })

df_all_results = pd.DataFrame(all_records)
print(f"Total Model Terevaluasi: {len(df_all_results)} berkas JSON.")
print("\nContoh 10 Konfigurasi Teratas Berdasarkan Akurasi:")
print(df_all_results.sort_values(by='Accuracy (%)', ascending=False).head(10).to_string(index=False))

Total Model Terevaluasi: 28 berkas JSON.

Contoh 10 Konfigurasi Teratas Berdasarkan Akurasi:
Classifier       Feature Domain  Dimension  Accuracy (%)  F1-Score (%)  Train Time (min)  Latency (ms)  RAM (MB)  Model Size (MB)
        LR vit-face-emotion-age       2304       88.6600       88.6800           54.7800        0.0046   18.9800           0.1100
       SVM vit-face-emotion-age       2304       88.6100       88.6300          511.1000        4.6785   18.9800          36.6000
       SVM     vit-emotion-face       1536       88.4300       88.4400          309.6000        0.9490   12.6600          15.2700
       SVM         vit-face-age       1536       87.5500       87.5700          492.3500        2.5468   12.6600          23.4800
        LR     vit-emotion-face       1536       87.4500       87.4900           32.3500        0.0035   12.6600           0.0800
        LR         vit-face-age       1536       87.4100       87.4000           30.4200        0.0056   12.6600           0.11

---
## 5. Profil Durasi Ekstraksi Fitur ViT GPU vs CPU (Table XI Naskah IEEE Access)

Kami membaca berkas `times/features_vit-*.json` untuk mengevaluasi durasi ekstraksi per citra dan total waktu ekstraksi untuk 10.800 citra pada kartu grafis NVIDIA RTX 4060 vs estimasi baseline CPU sekuensial.

Data ini mereproduksi **Table XI** pada naskah IEEE Access.

In [7]:
times_dir = 'times'
time_files = {
    'Biometric Face': 'features_vit-face.json',
    'Facial Emotion': 'features_vit-emotion.json',
    'Facial Age': 'features_vit-age.json'
}

extraction_records = []
total_gpu_time_s = 0.0
total_gpu_ms_per_img = 0.0
cache_size_per_domain_mb = 30.8

for domain, fname in time_files.items():
    fpath = os.path.join(times_dir, fname)
    with open(fpath, 'r') as fp:
        td = json.load(fp)
    
    t_sec = td['total_time_seconds']
    ms_img = td['avg_time_per_image_ms']
    total_gpu_time_s += t_sec
    total_gpu_ms_per_img += ms_img
    
    mins = int(t_sec // 60)
    secs = int(t_sec % 60)
    
    extraction_records.append({
        'Feature Domain': domain,
        'Avg Time / Image (ms)': round(ms_img, 2),
        'Total Time (s)': round(t_sec, 2),
        'Formatted GPU Time': f"{mins}m {secs:02d}s",
        'Cache Size on Disk (MB)': cache_size_per_domain_mb
    })

# Baris Fusi Tri-Domain
tri_mins = int(total_gpu_time_s // 60)
tri_secs = int(total_gpu_time_s % 60)
extraction_records.append({
    'Feature Domain': 'Tri-Domain Fusion (Face + Emotion + Age)',
    'Avg Time / Image (ms)': round(total_gpu_ms_per_img, 2),
    'Total Time (s)': round(total_gpu_time_s, 2),
    'Formatted GPU Time': f"{tri_mins}m {tri_secs:02d}s",
    'Cache Size on Disk (MB)': round(cache_size_per_domain_mb * 3, 1)
})

df_extraction = pd.DataFrame(extraction_records)
print("=== TABEL XI: PROFIL DURASI EKSTRAKSI FITUR VIT (10.800 CITRA) ===")
print(df_extraction.to_string(index=False))

=== TABEL XI: PROFIL DURASI EKSTRAKSI FITUR VIT (10.800 CITRA) ===
                          Feature Domain  Avg Time / Image (ms)  Total Time (s) Formatted GPU Time  Cache Size on Disk (MB)
                          Biometric Face                39.7400        429.2200             7m 09s                  30.8000
                          Facial Emotion                39.6500        428.2400             7m 08s                  30.8000
                              Facial Age                45.1100        487.1700             8m 07s                  30.8000
Tri-Domain Fusion (Face + Emotion + Age)               124.5000       1344.6400            22m 24s                  92.4000


---
## 6. Reproduksi Mandiri Table XII Naskah IEEE Access & Rekapitulasi Akhir

Di bawah ini adalah reproduksi mandiri **Table XII** yang menyatukan seluruh lima dimensi efisiensi komputasi:
1. **Model & Feature Configuration**
2. **Accuracy (%) & F1-Score (%)**
3. **Training Time (min)**
4. **Inference Latency (ms/sample)**
5. **RAM Consumption (MB)** (Jejak memori array data uji $2.160 \times D$ float32)
6. **Model Size (MB)** (Ukuran serialisasi biner model pada penyimpanan disk)

In [8]:
table_xii_data = [
    {
        'Classifier': 'Logistic Regression',
        'Feature Configuration': 'Tri-Domain (Face + Emotion + Age)',
        'Accuracy (%)': 88.66,
        'F1-Score (%)': 88.68,
        'Training Time (min)': 54.78,
        'Inference Latency (ms/sample)': 0.0046,
        'RAM Consumption (MB)': 18.98,
        'Model Size (MB)': 0.11
    },
    {
        'Classifier': 'Support Vector Machine',
        'Feature Configuration': 'Tri-Domain (Face + Emotion + Age)',
        'Accuracy (%)': 88.61,
        'F1-Score (%)': 88.63,
        'Training Time (min)': 511.10,
        'Inference Latency (ms/sample)': 4.6785,
        'RAM Consumption (MB)': 18.98,
        'Model Size (MB)': 36.60
    },
    {
        'Classifier': 'Random Forest',
        'Feature Configuration': 'Tri-Domain (Face + Emotion + Age)',
        'Accuracy (%)': 85.00,
        'F1-Score (%)': 84.93,
        'Training Time (min)': 110.47,
        'Inference Latency (ms/sample)': 0.0334,
        'RAM Consumption (MB)': 18.98,
        'Model Size (MB)': 6.19
    },
    {
        'Classifier': 'Gaussian Naive Bayes',
        'Feature Configuration': 'Dual-Domain (Emotion + Face)',
        'Accuracy (%)': 84.31,
        'F1-Score (%)': 84.30,
        'Training Time (min)': 25.49,
        'Inference Latency (ms/sample)': 0.0088,
        'RAM Consumption (MB)': 12.66,
        'Model Size (MB)': 0.78
    }
]

df_table_xii = pd.DataFrame(table_xii_data)
print("=== TABLE XII (REPRODUKSI RESMI IEEE ACCESS) ===")
print(df_table_xii.to_string(index=False))

print("\n=== KESIMPULAN EVALUASI EFISIENSI KOMPUTASI ===")
print("1. Konsistensi Satuan: Nilai RAM 18,98 MB pada naskah merepresentasikan 18,9844 MiB (kapasitas 2.160 x 2.304 x 4 bytes / 1024^2), sedangkan nilai desimalnya adalah 19,91 MB.")
print("2. Integritas Model: Eksperimen classifier menggunakan pipeline Scikit-learn murni; model_size_mb mencerminkan ukuran file serialisasi joblib di disk.")
print("3. Profiling RAM: Alokasi array data uji held-out pada RAM fisik (psutil) diverifikasi secara presisi melacak kapasitas teoritis .nbytes.")
print("4. Keunggulan Optimal: Logistic Regression (Tri-Domain) menawarkan trade-off komputasi terbaik dengan akurasi tertinggi (88,66%), model teringan (0,11 MB), dan latensi tercepat (0,0046 ms/sampel).")

=== TABLE XII (REPRODUKSI RESMI IEEE ACCESS) ===
            Classifier             Feature Configuration  Accuracy (%)  F1-Score (%)  Training Time (min)  Inference Latency (ms/sample)  RAM Consumption (MB)  Model Size (MB)
   Logistic Regression Tri-Domain (Face + Emotion + Age)       88.6600       88.6800              54.7800                         0.0046               18.9800           0.1100
Support Vector Machine Tri-Domain (Face + Emotion + Age)       88.6100       88.6300             511.1000                         4.6785               18.9800          36.6000
         Random Forest Tri-Domain (Face + Emotion + Age)       85.0000       84.9300             110.4700                         0.0334               18.9800           6.1900
  Gaussian Naive Bayes      Dual-Domain (Emotion + Face)       84.3100       84.3000              25.4900                         0.0088               12.6600           0.7800

=== KESIMPULAN EVALUASI EFISIENSI KOMPUTASI ===
1. Konsistensi Satuan: